**该文档为 AI 生成，但未来需要人工作修改**

# DFT XC skeleton 二阶导数分解 (TPSS0, MGGA)

本文档对标 `02-2-decomp_de_J.ipynb` 与 `02-3-decomp_de_K.ipynb`：将 PySCF 中 `pyscf.hessian.rks._get_vxc_diag` 与 `_get_vxc_deriv2` 的功能以较为简明的形式展开，使得每一项的数学表达式都能直接对应到代码片段。

**边界条件**：

- 仅考虑泛函在固定格点上的二阶导数 (skeleton)，**不考虑** grid response (即不考虑格点权重 / 格点坐标对 $\mathbf{R}_A$ 的依赖)。这对应 PySCF 中 `grids_response = False` 的默认行为。
- 仅使用 `eval_xc_eff` (而非 `eval_xc`) 获取 $v_{xc}$ 与 $f_{xc}$。
- 不使用 `ao_loc`、`non0tab` / `mask` 等优化。所有 AO 在所有格点上都参与运算。
- 不使用 PySCF 的 `_scale_ao` / `_dot_ao_ao` / `_dot_ao_dm` / `_make_dR_dao_w` / `_d1d2_dot_` / `_make_dR_rho1` 等抽象函数，所有缩并直接以 `np.einsum` 写出。
- 目标泛函 TPSS0 是 hybrid meta-GGA，对应组分 `[rho, grad_x, grad_y, grad_z, tau]` (共 5 个 component)。

In [1]:
from pyscf import gto, dft, lib
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = dft.RKS(mol, xc="TPSS0").density_fit()
dat0 = np.load("nh3_r_tpss0.npz")
mf.mo_coeff = dat0["mo_coeff"]
mf.mo_occ = dat0["mo_occ"]
mf.mo_energy = dat0["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
# Reference de_vxc from 06-1: this is what we want to reproduce.
de_vxc_ref = np.load("nh3_r_tpss0_decomp.npz")["de_vxc"]
print("de_vxc_ref shape:", de_vxc_ref.shape)
print("de_vxc_ref fp:   ", lib.fp(de_vxc_ref))

de_vxc_ref shape: (4, 4, 3, 3)
de_vxc_ref fp:    -0.8310821568111217


## 准备：格点、密度、$v_{xc}$、$f_{xc}$

XC 数值积分的能量表达为

$$
E_{xc} = \sum_g w_g \; \epsilon_{xc}\bigl(\boldsymbol\rho(g)\bigr)
$$

其中 $g$ 标记格点，$w_g$ 为格点权重；$\boldsymbol\rho(g)$ 为密度的“组分向量” (LDA 时为 $\rho$；GGA 时为 $[\rho, \partial_x\rho, \partial_y\rho, \partial_z\rho]$；MGGA 时再加 $\tau$)。

对 MGGA (无 laplacian)：

- $\boldsymbol\rho = [\rho, \partial_x\rho, \partial_y\rho, \partial_z\rho, \tau]$，共 5 个分量；
- $\tau = \tfrac12 \sum_v (\partial_v\phi_\alpha)(\partial_v\phi_\beta) D_{\alpha\beta}$；
- $v_{xc}[c, g] := \partial \epsilon_{xc} / \partial \rho_c$，$f_{xc}[c, c', g] := \partial^2 \epsilon_{xc} / \partial \rho_c \partial \rho_{c'}$。

`eval_xc_eff` 直接返回上述定义下的 $v_{xc}$ 与 $f_{xc}$。

In [5]:
# Build grid (no grid response means we just need coords + weights).
grids = mf.grids
if grids.coords is None:
    grids.build(with_non0tab=False)

ni = mf._numint
xctype = ni._xc_type(mf.xc)
print("xctype:", xctype)
assert xctype == "MGGA"  # this notebook is specifically for MGGA

xctype: MGGA


In [6]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mocc = mo_coeff[:, mo_occ > 0]
dm0 = mocc @ mocc.T * 2
natm = mol.natm
nao = mol.nao
aoslices = mol.aoslice_by_atom()

我们一次性计算所有格点上的 AO 一阶到三阶导数。 PySCF 的 `eval_ao(mol, coords, deriv=3)` 返回形状为 `[20, ngrid, nao]` 的数组，其中第 0 维的分量含义如下：

| index | derivative | index | derivative |
|:---:|:---:|:---:|:---:|
| 0 | $\phi$        | 10 | $\partial_{xxx}\phi$ |
| 1 | $\partial_x\phi$ | 11 | $\partial_{xxy}\phi$ |
| 2 | $\partial_y\phi$ | 12 | $\partial_{xxz}\phi$ |
| 3 | $\partial_z\phi$ | 13 | $\partial_{xyy}\phi$ |
| 4 | $\partial_{xx}\phi$ | 14 | $\partial_{xyz}\phi$ |
| 5 | $\partial_{xy}\phi$ | 15 | $\partial_{xzz}\phi$ |
| 6 | $\partial_{xz}\phi$ | 16 | $\partial_{yyy}\phi$ |
| 7 | $\partial_{yy}\phi$ | 17 | $\partial_{yyz}\phi$ |
| 8 | $\partial_{yz}\phi$ | 18 | $\partial_{yzz}\phi$ |
| 9 | $\partial_{zz}\phi$ | 19 | $\partial_{zzz}\phi$ |

为了让索引与方向 $(t, s) \in \{x, y, z\}^2$ 对应，下面定义两个辅助函数。

In [7]:
# 2nd-derivative index table: ao_idx2[t][s] -> index in 4..9 for d_t d_s phi
ao_idx2 = np.array([[4, 5, 6], [5, 7, 8], [6, 8, 9]])

# 3rd-derivative index table: ao_idx3[t][s][v] -> index in 10..19 for d_t d_s d_v phi
_table3 = {
    (0, 0, 0): 10, (0, 0, 1): 11, (0, 0, 2): 12,
    (0, 1, 1): 13, (0, 1, 2): 14, (0, 2, 2): 15,
    (1, 1, 1): 16, (1, 1, 2): 17, (1, 2, 2): 18,
    (2, 2, 2): 19,
}
ao_idx3 = np.zeros((3, 3, 3), dtype=int)
for t in range(3):
    for s in range(3):
        for v in range(3):
            ao_idx3[t, s, v] = _table3[tuple(sorted([t, s, v]))]

# Evaluate AOs (up to 3rd derivative) on all grid points.
ao = ni.eval_ao(mol, grids.coords, deriv=3)  # shape [20, ngrid, nao]
weight = grids.weights
ngrid = ao.shape[1]
print("ao shape:", ao.shape, "ngrid:", ngrid)

ao shape: (20, 43328, 49) ngrid: 43328


再用 `eval_rho2` 计算自洽密度对应的 $\boldsymbol\rho$，然后用 `eval_xc_eff` 取到 $v_{xc}, f_{xc}$。

In [8]:
rho = ni.eval_rho2(mol, ao[:10], mo_coeff, mo_occ, xctype=xctype)
# rho shape: [6, ngrid] = [rho, gx, gy, gz, lap, tau]; for vxc/fxc only 5 components are used.
xc_eff = ni.eval_xc_eff(mf.xc, rho, deriv=2, xctype=xctype)
vxc = xc_eff[1]  # shape [5, ngrid]
fxc = xc_eff[2]  # shape [5, 5, ngrid]
print("vxc shape:", vxc.shape, "fxc shape:", fxc.shape)

vxc shape: (5, 43328) fxc shape: (5, 5, 43328)


## 分解策略

对于电子能量 $E_{xc}$ 关于核坐标 $\mathbf R_{A,t}$ 与 $\mathbf R_{B,s}$ 的二阶 skeleton 导数：

$$
\frac{\partial^2 E_{xc}}{\partial \mathbf R_{A,t} \partial \mathbf R_{B,s}}
= \sum_g w_g \Bigl[ \sum_c v_{xc,c}\;\rho^{(2)}_{c}[A,t;B,s] + \sum_{cc'} \rho^{(1)}_{c}[A,t]\; f_{xc,cc'}\;\rho^{(1)}_{c'}[B,s] \Bigr]
$$

其中：

- $\rho^{(1)}_c[A,t](g)$：密度分量 $\rho_c$ 关于 $\mathbf R_{A,t}$ 的一阶 skeleton 导数；
- $\rho^{(2)}_c[A,t;B,s](g)$：密度分量 $\rho_c$ 关于 $\mathbf R_{A,t}$ 与 $\mathbf R_{B,s}$ 的混合二阶 skeleton 导数。

skeleton 导数的约定 (与 02-2 一致)：

$$
\frac{\partial \phi_\alpha(g)}{\partial \mathbf R_{A,t}} = -\delta_{\alpha \in A} \,(\partial_t \phi_\alpha)(g)
$$

即把核 $A$ 沿 $t$ 方向移动等价于把 $A$ 上所有基函数沿 $t$ 反向求导。

为了写起来与 PySCF 习惯一致，我们记 $\tilde\rho^{(1)}_c[A,t] := -\partial_{\mathbf R_{A,t}} \rho_c$ (取正号)；$\tilde\rho^{(2)}_c[A,t;B,s] := +\partial_{\mathbf R_{A,t}}\partial_{\mathbf R_{B,s}} \rho_c$。则

$$
\frac{\partial^2 E_{xc}}{\partial \mathbf R_{A,t} \partial \mathbf R_{B,s}}
= \sum_g w_g \Bigl[ \sum_c v_{xc,c}\;\tilde\rho^{(2)}_{c}[A,t;B,s] + \sum_{cc'} \tilde\rho^{(1)}_{c}[A,t]\; f_{xc,cc'}\;\tilde\rho^{(1)}_{c'}[B,s] \Bigr]
$$

(两个负号在 $\tilde\rho^{(1)}$ 上互相抵消；$\tilde\rho^{(2)}$ 本身与 $\rho^{(2)}$ 同号。)

我们的目标即为：

1. 计算 `rho1[A, t, c, g]` $= \tilde\rho^{(1)}_c[A,t](g)$；
2. 计算 `rho2[A, B, t, s, c, g]` $= \tilde\rho^{(2)}_c[A,t;B,s](g)$；
3. 分别得到 `de_vxc_fxc` 与 `de_vxc_rho2` 两部分，相加。

## $\tilde\rho^{(1)}$：密度的一阶 skeleton 导数

为了缩并方便，先预先计算半变换矩阵

$$
\text{ao\_dm0}_c[g, \beta] := \sum_\alpha (\partial_c \phi_\alpha)(g)\, D_{\alpha\beta}
$$

其中 $\partial_c$ 表示对 $c$ 方向的偏导 ($c = 0,1,2,3$ 分别对应不求导、$\partial_x, \partial_y, \partial_z$)。

In [9]:
ao_dm0 = np.einsum("cgu,uv->cgv", ao[:4], dm0)  # shape [4, ngrid, nao]

**(a) $c = 0$ (即 $\rho$ 本身)**

$\rho = \sum_{\alpha\beta} \phi_\alpha \phi_\beta D_{\alpha\beta}$。对 $\mathbf R_{A,t}$ 求一次导：

$$
\tilde\rho^{(1)}_0[A,t] = 2 \sum_{\alpha \in A, \beta} (\partial_t \phi_\alpha) \phi_\beta D_{\alpha\beta}
  = 2 \sum_{\alpha \in A} (\partial_t \phi_\alpha)(g) \;\text{ao\_dm0}_0[g, \alpha]
$$

其中因子 2 来自 $\alpha$ 与 $\beta$ 的对称性 (任一在 $A$ 上都贡献，结果等价于其中一个限定在 $A$ 上乘 2)。

**(b) $c = 1, 2, 3$ (即 $\partial_u \rho$)**

$\partial_u \rho = \sum_{\alpha\beta} [(\partial_u\phi_\alpha) \phi_\beta + \phi_\alpha (\partial_u\phi_\beta)] D_{\alpha\beta} = 2\sum (\partial_u\phi_\alpha) \phi_\beta D_{\alpha\beta}$ (由对称性合并)。求 $\mathbf R_{A,t}$ 导后：

$$
\tilde\rho^{(1)}_{1+u}[A,t] = 2 \sum_{\alpha \in A, \beta} \bigl[ (\partial_t \partial_u \phi_\alpha) \phi_\beta + (\partial_t \phi_\alpha)(\partial_u \phi_\beta) \bigr] D_{\alpha\beta}
$$

用 ao\_dm0 写：

$$
\tilde\rho^{(1)}_{1+u}[A,t] = 2 \sum_{\alpha \in A} \bigl[ (\partial_t \partial_u \phi_\alpha)(g)\;\text{ao\_dm0}_0[g, \alpha] + (\partial_t \phi_\alpha)(g)\;\text{ao\_dm0}_{1+u}[g, \alpha] \bigr]
$$

**(c) $c = 4$ (即 $\tau$)**

$\tau = \tfrac12 \sum_v (\partial_v\phi_\alpha)(\partial_v\phi_\beta) D_{\alpha\beta}$。求 $\mathbf R_{A,t}$ 导后：

$$
\tilde\rho^{(1)}_4[A,t] = \sum_v \sum_{\alpha \in A,\beta} (\partial_t \partial_v \phi_\alpha)(\partial_v \phi_\beta) D_{\alpha\beta}
  = \sum_v \sum_{\alpha \in A} (\partial_t \partial_v \phi_\alpha)(g)\;\text{ao\_dm0}_{1+v}[g, \alpha]
$$

(对称性已消去 $\tfrac12$。)

In [10]:
rho1 = np.zeros((natm, 3, 5, ngrid))
for A in range(natm):
    p0, p1 = aoslices[A][2:]
    # c = 0: rho
    for t in range(3):
        rho1[A, t, 0] = 2 * np.einsum(
            "gi,gi->g",
            ao[1 + t, :, p0:p1],
            ao_dm0[0, :, p0:p1],
        )
    # c = 1+u: grad_u rho
    for t in range(3):
        for u in range(3):
            term1 = np.einsum("gi,gi->g", ao[ao_idx2[t, u], :, p0:p1], ao_dm0[0, :, p0:p1])
            term2 = np.einsum("gi,gi->g", ao[1 + t, :, p0:p1], ao_dm0[1 + u, :, p0:p1])
            rho1[A, t, 1 + u] = 2 * (term1 + term2)
    # c = 4: tau
    for t in range(3):
        tau_t = np.zeros(ngrid)
        for v in range(3):
            tau_t += np.einsum("gi,gi->g", ao[ao_idx2[t, v], :, p0:p1], ao_dm0[1 + v, :, p0:p1])
        rho1[A, t, 4] = tau_t
print("rho1 fp:", lib.fp(rho1))

rho1 fp: 16948168.18739759


## 第一部分：$\rho^{(1)} \cdot f_{xc} \cdot \rho^{(1)}$ 贡献

$$
\text{de\_vxc\_fxc}[A, B, t, s] = \sum_g w_g \sum_{cc'} \tilde\rho^{(1)}_c[A,t](g)\; f_{xc,cc'}(g)\; \tilde\rho^{(1)}_{c'}[B,s](g)
$$

由于我们的 $\tilde\rho^{(1)}$ 已经包含 $\tau$ 的所有因子（即没有再单独保留 $\tfrac12$），$f_{xc}$ 内 $\tau$ 维度的乘法对称已经自然成立。

In [11]:
de_vxc_fxc = np.einsum("g,Atcg,cdg,Bsdg->ABts", weight, rho1, fxc, rho1)
print("de_vxc_fxc fp:", lib.fp(de_vxc_fxc))

de_vxc_fxc fp: -29.390069496788392


## 第二部分：$v_{xc} \cdot \rho^{(2)}$ 贡献

对每个 $(A, B)$ 都需要 $\tilde\rho^{(2)}[A, B, t, s, c]$。skeleton 二阶导意味着两次 $-\partial_{\mathbf R}$ 都作用在 AO 上 (每个 AO 一次方向求导)。

为了清晰处理“同原子 $A=B$”与“异原子 $A \neq B$”，先按密度分量 $c$ 逐项给出表达式。下面统一记 $\alpha \in A$ 与 $\beta \in B$ 的限制条件，cross-AO 项对任意 $(A,B)$ 都有贡献，single-AO 项 (两次导都落在同一 AO 上) 只在 $A=B$ 时有贡献。

### (a) $c = 0$ ($\rho$)

$$
\tilde\rho^{(2)}_0[A,t;B,s] = 2 \sum_{\alpha\in A,\,\beta\in B} (\partial_t \phi_\alpha)(\partial_s \phi_\beta) D_{\alpha\beta}
  + [\![A=B]\!] \cdot 2 \sum_{\alpha \in A} (\partial_t \partial_s \phi_\alpha)\;\text{ao\_dm0}_0[g,\alpha]
$$

### (b) $c = 1+u$ ($\partial_u \rho$)

对 $\partial_u \rho = 2 \sum (\partial_u \phi_\alpha) \phi_\beta D_{\alpha\beta}$ 做两次 R 求导：

$$
\tilde\rho^{(2)}_{1+u}[A,t;B,s] = 2 \sum_{\alpha\in A,\,\beta\in B} \bigl[(\partial_t\partial_u \phi_\alpha)(\partial_s \phi_\beta) + (\partial_t \phi_\alpha)(\partial_s\partial_u \phi_\beta)\bigr] D_{\alpha\beta}
$$

$$
\quad + [\![A=B]\!] \cdot 2 \sum_{\alpha \in A} \bigl[(\partial_t\partial_s\partial_u \phi_\alpha)\;\text{ao\_dm0}_0[g,\alpha] + (\partial_t\partial_s \phi_\alpha)\;\text{ao\_dm0}_{1+u}[g,\alpha]\bigr]
$$

### (c) $c = 4$ ($\tau$)

对 $\tau = \tfrac12 \sum_v (\partial_v\phi_\alpha)(\partial_v\phi_\beta) D_{\alpha\beta}$，两次 R 导分配方式：

- cross-AO ($A,t$ 落在 $\alpha$，$B,s$ 落在 $\beta$；以及 $\alpha,\beta$ 互换的对称项，二者由 $D$ 对称合并)；
- single-AO ($A=B$，两次导都落在同一 $\alpha$ 上)。

结果：

$$
\tilde\rho^{(2)}_4[A,t;B,s] = \sum_v \sum_{\alpha\in A,\beta\in B} (\partial_t\partial_v \phi_\alpha)(\partial_s\partial_v \phi_\beta) D_{\alpha\beta}
  + [\![A=B]\!] \cdot \sum_v \sum_{\alpha\in A} (\partial_t\partial_s\partial_v \phi_\alpha)\;\text{ao\_dm0}_{1+v}[g,\alpha]
$$

**注意 cross-AO 项不带 $\tfrac12$**：因为 $\alpha,\beta$ 对应不同原子时，$D$ 的对称性已经把两个对称组合 (case 3a + 3b) 合并成一项。

另外注意 cross-AO 在 $A \neq B$ 时不强制 $(t,s)$ 对称化；最终 Hessian 的 $(t,s) \leftrightarrow (s,t)$ 对称性由 $\text{de}[B,A] = \text{de}[A,B]^\mathsf{T}$ 在外层组装时保证。

In [12]:
de_vxc_rho2 = np.zeros((natm, natm, 3, 3))
for A in range(natm):
    pA, qA = aoslices[A][2:]
    sA = slice(pA, qA)
    for B in range(natm):
        pB, qB = aoslices[B][2:]
        sB = slice(pB, qB)

        rho2 = np.zeros((5, 3, 3, ngrid))

        # ----- (a) c = 0 -----
        for t in range(3):
            for s in range(3):
                # cross-AO
                rho2[0, t, s] += 2 * np.einsum(
                    "gi,ij,gj->g",
                    ao[1 + t, :, sA], dm0[sA, sB], ao[1 + s, :, sB],
                )
                # single-AO (only A == B)
                if A == B:
                    rho2[0, t, s] += 2 * np.einsum(
                        "gi,gi->g",
                        ao[ao_idx2[t, s], :, sA], ao_dm0[0, :, sA],
                    )

        # ----- (b) c = 1+u -----
        for t in range(3):
            for s in range(3):
                for u in range(3):
                    # cross-AO
                    rho2[1 + u, t, s] += 2 * np.einsum(
                        "gi,ij,gj->g",
                        ao[ao_idx2[t, u], :, sA], dm0[sA, sB], ao[1 + s, :, sB],
                    )
                    rho2[1 + u, t, s] += 2 * np.einsum(
                        "gi,ij,gj->g",
                        ao[1 + t, :, sA], dm0[sA, sB], ao[ao_idx2[s, u], :, sB],
                    )
                    # single-AO (only A == B)
                    if A == B:
                        rho2[1 + u, t, s] += 2 * np.einsum(
                            "gi,gi->g",
                            ao[ao_idx3[t, s, u], :, sA], ao_dm0[0, :, sA],
                        )
                        rho2[1 + u, t, s] += 2 * np.einsum(
                            "gi,gi->g",
                            ao[ao_idx2[t, s], :, sA], ao_dm0[1 + u, :, sA],
                        )

        # ----- (c) c = 4 (tau) -----
        for t in range(3):
            for s in range(3):
                # cross-AO
                for v in range(3):
                    rho2[4, t, s] += np.einsum(
                        "gi,ij,gj->g",
                        ao[ao_idx2[t, v], :, sA], dm0[sA, sB], ao[ao_idx2[s, v], :, sB],
                    )
                # single-AO (only A == B)
                if A == B:
                    for v in range(3):
                        rho2[4, t, s] += np.einsum(
                            "gi,gi->g",
                            ao[ao_idx3[t, s, v], :, sA], ao_dm0[1 + v, :, sA],
                        )

        # contract with v_xc and weight
        de_vxc_rho2[A, B] = np.einsum("g,cg,ctsg->ts", weight, vxc, rho2)

print("de_vxc_rho2 fp:", lib.fp(de_vxc_rho2))

de_vxc_rho2 fp: 28.558987339976962


## 总核验

$$
\text{de\_vxc} = \underbrace{\text{de\_vxc\_rho2}}_{v_{xc}\cdot \tilde\rho^{(2)}} + \underbrace{\text{de\_vxc\_fxc}}_{\tilde\rho^{(1)}\cdot f_{xc}\cdot \tilde\rho^{(1)}}
$$

In [13]:
de_vxc_recap = de_vxc_rho2 + de_vxc_fxc
print("de_vxc_recap fp:", lib.fp(de_vxc_recap))
print("de_vxc_ref   fp:", lib.fp(de_vxc_ref))
print("max abs diff   :", np.max(np.abs(de_vxc_recap - de_vxc_ref)))
assert np.allclose(de_vxc_recap, de_vxc_ref, atol=1e-10)

de_vxc_recap fp: -0.8310821568113991
de_vxc_ref   fp: -0.8310821568111217
max abs diff   : 4.291678123991005e-12
